# GraphMS-Net — Segmentation Evaluation

**Purpose:** rapid, no-retraining validation of the frozen GraphMS v3.5.1 Hybrid segmentation system.

This notebook does **not** train, tune, select thresholds, or alter the frozen pipeline. It uses the committed Stage16 per-case table and the already-generated final hybrid masks to produce qualitative evaluation evidence: **FLAIR | expert ground truth | final prediction | TP/FP/FN error map** for representative best, median, and worst development cases.

Claim scope remains **five-fold development cross-validation**; this is not external clinical validation.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, sys
REPO = '/content/GraphMS-Net'
if not os.path.isdir(os.path.join(REPO, '.git')):
    subprocess.run(['git','clone','-q','https://github.com/sath17-o/GraphMS-Net.git',REPO], check=True)
subprocess.run(['git','-C',REPO,'pull','-q','--ff-only'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','nibabel'], check=True)
print('Repository and dependencies ready.')


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

METRICS = Path(REPO) / 'results/stage16/STAGE16_PER_CASE_METRICS.csv'
df = pd.read_csv(METRICS)
assert len(df) == 93, f'Expected 93 final development cases, found {len(df)}'
print('Cases:', len(df), '| folds:', sorted(df.fold.unique().tolist()))
print('DSC range:', float(df.DSC.min()), 'to', float(df.DSC.max()))


## Representative-case selection

Cases are selected **deterministically from the frozen Stage16 DSC column**. No case is hand-picked by visual appearance. The median case is the row closest to the cohort median DSC.


In [ ]:
best = df.loc[df.DSC.idxmax()]
worst = df.loc[df.DSC.idxmin()]
median_value = float(df.DSC.median())
median = df.loc[(df.DSC - median_value).abs().idxmin()]
selected = pd.DataFrame([best, median, worst], index=['BEST','MEDIAN','WORST'])
display(selected[['case','fold','DSC','IoU','Sensitivity','Specificity','HD95_mm','TP','FP','FN']])


In [ ]:
def load_bool(path):
    return np.asarray(nib.load(str(path)).dataobj) > 0

def robust_flair(path):
    x = np.asarray(nib.load(str(path)).dataobj, dtype=np.float32)
    finite = np.isfinite(x)
    if finite.any():
        lo, hi = np.percentile(x[finite], [1, 99])
        x = np.clip((x-lo)/(hi-lo+1e-8), 0, 1)
    return x

def choose_slice(gt, pred):
    burden = np.sum(gt | pred, axis=(0,1))
    return int(np.argmax(burden))

def show_case(row, label):
    case = row['case']
    gt_path = Path(row['gt_path'])
    pred_path = Path(row['mask_path'])
    flair_path = gt_path.parent.parent / 'imagesTr' / f'{case}_0000.nii.gz'
    for p in [gt_path, pred_path, flair_path]:
        assert p.exists(), f'Missing required file: {p}'

    gt = load_bool(gt_path)
    pred = load_bool(pred_path)
    flair = robust_flair(flair_path)
    assert gt.shape == pred.shape == flair.shape, (gt.shape, pred.shape, flair.shape)
    z = choose_slice(gt, pred)

    tp = gt & pred
    fp = (~gt) & pred
    fn = gt & (~pred)
    err = np.zeros(gt.shape, dtype=np.uint8)
    err[tp] = 1; err[fp] = 2; err[fn] = 3

    fig, ax = plt.subplots(1,4,figsize=(16,4), constrained_layout=True)
    for a in ax:
        a.imshow(flair[:,:,z].T, cmap='gray', origin='lower')
        a.axis('off')
    ax[0].set_title(f'{label}: {case}\nFLAIR — slice {z}')
    ax[1].imshow(np.ma.masked_where(~gt[:,:,z].T, gt[:,:,z].T), cmap='autumn', alpha=.65, origin='lower')
    ax[1].set_title('Expert ground truth')
    ax[2].imshow(np.ma.masked_where(~pred[:,:,z].T, pred[:,:,z].T), cmap='winter', alpha=.65, origin='lower')
    ax[2].set_title('Frozen final prediction')
    cmap = ListedColormap(['black','lime','red','deepskyblue'])
    ax[3].imshow(np.ma.masked_where(err[:,:,z].T==0, err[:,:,z].T), cmap=cmap, vmin=0, vmax=3, alpha=.8, origin='lower')
    ax[3].set_title('Error map: TP green | FP red | FN blue')
    fig.suptitle(f"DSC {row['DSC']:.4f} | IoU {row['IoU']:.4f} | Sens {row['Sensitivity']:.4f} | Spec {row['Specificity']:.6f} | HD95 {row['HD95_mm']:.2f} mm", fontsize=11)
    out = Path('/content') / f"SEGMENTATION_{label}_{case}.png"
    fig.savefig(out, dpi=220, bbox_inches='tight')
    plt.show()
    return out

outputs=[]
for label, row in selected.iterrows():
    outputs.append(show_case(row, label))
print('Saved:', *outputs, sep='\n - ')


## Interpretation

The three panels expose both strengths and failure modes without changing the frozen system. **TP** shows correctly segmented lesion voxels, **FP** shows predicted lesion voxels absent from the expert mask, and **FN** shows expert lesion voxels missed by the model.

The final system remains: ResEncM-250 → graph construction → TrueGAT → CNN/GNN hybrid fusion → decoder/head → cross-fitted Stage11. Fold-specific Stage11 thresholds remain frozen (0.40 for folds 0/1/3/4; 0.45 for fold 2), with 26-connectivity and a 10-voxel minimum component size. No morphology is introduced by this analysis.


## Cohort-level segmentation diagnostics

This section uses only the **committed 93-case Stage16 table**. It performs no inference, training, tuning, threshold selection, or model modification. Ground-truth lesion burden is used **only for retrospective error analysis**, never as an inference input.

Two summaries are deliberately kept separate:

- **Official Stage16 result:** equal-weight mean of the five outer-fold case means, with sample SD across folds.
- **Case-level descriptive result:** ordinary statistics across all 93 cases, used only to understand the distribution and failure modes.


In [ ]:
audit = df.copy()

audit['GT_lesion_voxels'] = audit['TP'] + audit['FN']
audit['Pred_lesion_voxels'] = audit['TP'] + audit['FP']
audit['Precision'] = audit['TP'] / (audit['TP'] + audit['FP']).replace(0, np.nan)
audit['FNR'] = audit['FN'] / (audit['TP'] + audit['FN']).replace(0, np.nan)
audit['Dominant_error'] = np.where(audit['FN'] > audit['FP'], 'FN-dominant', 'FP-dominant')

metric_cols = ['DSC','IoU','Sensitivity','Specificity','HD95_mm']
fold_case_means = audit.groupby('fold', sort=True)[metric_cols].mean()
official_mean = fold_case_means.mean()
official_sd = fold_case_means.std(ddof=1)

summary = pd.DataFrame({
    'value': {
        'cases': len(audit),
        'official_equal_fold_DSC_mean': official_mean['DSC'],
        'official_equal_fold_DSC_sd': official_sd['DSC'],
        'case_level_DSC_mean': audit['DSC'].mean(),
        'case_level_DSC_median': audit['DSC'].median(),
        'case_level_DSC_min': audit['DSC'].min(),
        'case_level_DSC_max': audit['DSC'].max(),
        'case_level_HD95_median_mm': audit['HD95_mm'].median(),
        'case_level_sensitivity_median': audit['Sensitivity'].median(),
        'case_level_precision_median': audit['Precision'].median(),
        'cases_DSC_lt_0_50': int((audit['DSC'] < 0.50).sum()),
        'cases_DSC_lt_0_60': int((audit['DSC'] < 0.60).sum()),
        'cases_DSC_ge_0_80': int((audit['DSC'] >= 0.80).sum()),
        'cases_HD95_gt_20mm': int((audit['HD95_mm'] > 20).sum()),
        'cases_sensitivity_lt_0_50': int((audit['Sensitivity'] < 0.50).sum()),
    }
})

print('OFFICIAL STAGE16 — equal-weight mean of five outer-fold case means')
display(pd.DataFrame({'mean': official_mean, 'sd_across_folds': official_sd}))
print('CASE-LEVEL DESCRIPTIVE DIAGNOSTICS — 93 development cases')
display(summary)

summary.to_csv('/content/SEGMENTATION_COHORT_SUMMARY.csv')
fold_case_means.to_csv('/content/SEGMENTATION_FOLD_CASE_MEANS.csv')


### Distribution and fold diagnostics

The following plots are descriptive. They are intended to show whether the headline mean is supported consistently across cases and folds, and to make outlier behavior visible.


In [ ]:
# 1) DSC distribution
fig, ax = plt.subplots(figsize=(8,5))
ax.hist(audit['DSC'], bins=15)
ax.axvline(float(official_mean['DSC']), linestyle='--', label='Official equal-fold mean')
ax.axvline(float(audit['DSC'].median()), linestyle=':', label='Case median')
ax.set_xlabel('Dice Similarity Coefficient')
ax.set_ylabel('Number of cases')
ax.set_title('GraphMS v3.5.1 — DSC distribution across 93 development cases')
ax.legend()
fig.tight_layout()
fig.savefig('/content/SEGMENTATION_DSC_DISTRIBUTION.png', dpi=220, bbox_inches='tight')
plt.show()

# 2) HD95 distribution
fig, ax = plt.subplots(figsize=(8,5))
ax.hist(audit['HD95_mm'], bins=15)
ax.axvline(float(audit['HD95_mm'].median()), linestyle='--', label='Case median')
ax.set_xlabel('HD95 (mm)')
ax.set_ylabel('Number of cases')
ax.set_title('GraphMS v3.5.1 — HD95 distribution across 93 development cases')
ax.legend()
fig.tight_layout()
fig.savefig('/content/SEGMENTATION_HD95_DISTRIBUTION.png', dpi=220, bbox_inches='tight')
plt.show()

# 3) Fold-wise DSC — bind the boxplot to the saved figure explicitly.
fig, ax = plt.subplots(figsize=(8,5))
audit.boxplot(column='DSC', by='fold', grid=False, ax=ax)
fig.suptitle('')
ax.set_title('Fold-wise DSC distribution')
ax.set_xlabel('Outer fold')
ax.set_ylabel('DSC')
fig.tight_layout()
fig.savefig('/content/SEGMENTATION_DSC_BY_FOLD.png', dpi=220, bbox_inches='tight')
plt.show()


### Lesion-burden failure analysis

MS lesion segmentation can be disproportionately difficult when the expert lesion burden is small. This analysis tests that **descriptively** using only the frozen OOF confusion counts. It does not change the model and does not establish causation.


In [ ]:
# Deterministic rank quartiles. With 93 cases this yields 23/23/23/24,
# matching the frozen report tables and avoiding qcut's alternate 24/23/23/23 allocation.
labels = ['Q1 smallest','Q2','Q3','Q4 largest']
order = (
    audit[['GT_lesion_voxels','case']]
    .sort_values(['GT_lesion_voxels','case'], kind='mergesort')
    .index
    .to_list()
)
n_cases = len(order)
edges = [0, n_cases//4, (2*n_cases)//4, (3*n_cases)//4, n_cases]
quartile = pd.Series(index=audit.index, dtype='object')
for label, lo, hi in zip(labels, edges[:-1], edges[1:]):
    quartile.loc[order[lo:hi]] = label

audit['burden_quartile'] = pd.Categorical(
    quartile, categories=labels, ordered=True
)

burden_summary = (
    audit.groupby('burden_quartile', observed=True)
         .agg(
             n=('case','size'),
             GT_vox_min=('GT_lesion_voxels','min'),
             GT_vox_max=('GT_lesion_voxels','max'),
             DSC_mean=('DSC','mean'),
             DSC_median=('DSC','median'),
             Sensitivity_mean=('Sensitivity','mean'),
             HD95_median_mm=('HD95_mm','median'),
         )
)
assert burden_summary['n'].tolist() == [23, 23, 23, 24], burden_summary['n'].tolist()

r_logburden_dsc = float(np.corrcoef(np.log10(audit['GT_lesion_voxels'] + 1), audit['DSC'])[0,1])
print('Pearson r(log10 GT lesion voxels, DSC) =', round(r_logburden_dsc, 4))
display(burden_summary)
burden_summary.to_csv('/content/SEGMENTATION_BURDEN_QUARTILES.csv')

fig, ax = plt.subplots(figsize=(8,5))
ax.scatter(audit['GT_lesion_voxels'], audit['DSC'], alpha=0.75)
ax.set_xscale('log')
x = np.log10(audit['GT_lesion_voxels'].to_numpy() + 1)
y = audit['DSC'].to_numpy()
coef = np.polyfit(x, y, 1)
xs = np.linspace(x.min(), x.max(), 200)
ax.plot(10**xs - 1, np.polyval(coef, xs), linestyle='--')
ax.set_xlabel('Expert lesion burden (voxels, log scale)')
ax.set_ylabel('DSC')
ax.set_title('DSC versus expert lesion burden — descriptive development analysis')
fig.tight_layout()
fig.savefig('/content/SEGMENTATION_DSC_VS_LESION_BURDEN.png', dpi=220, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(8,5))
burden_summary['DSC_mean'].plot(kind='bar', ax=ax)
ax.set_ylabel('Mean DSC')
ax.set_xlabel('Ground-truth lesion-burden quartile')
ax.set_title('Mean DSC by expert lesion-burden quartile')
ax.tick_params(axis='x', rotation=0)
fig.tight_layout()
fig.savefig('/content/SEGMENTATION_DSC_BY_BURDEN_QUARTILE.png', dpi=220, bbox_inches='tight')
plt.show()


### Worst-case table and dominant error direction

The table below exposes the lowest-DSC cases rather than hiding them. `FN-dominant` means missed lesion voxels exceed false-positive lesion voxels for that case; `FP-dominant` means the reverse.


In [ ]:
worst10 = (
    audit.sort_values('DSC', ascending=True)
         .head(10)
         [['case','fold','DSC','IoU','Sensitivity','Precision','HD95_mm',
           'GT_lesion_voxels','TP','FP','FN','Dominant_error']]
         .reset_index(drop=True)
)

display(worst10)
worst10.to_csv('/content/SEGMENTATION_WORST10.csv', index=False)

low_dsc = audit[audit['DSC'] < 0.60]
print('Cases with DSC < 0.60:', len(low_dsc))
print('FN-dominant among DSC < 0.60:', int((low_dsc['Dominant_error'] == 'FN-dominant').sum()))
print('FP-dominant among DSC < 0.60:', int((low_dsc['Dominant_error'] == 'FP-dominant').sum()))

print('\nFrozen-table diagnostic summary')
print(f"- Official equal-fold DSC: {official_mean['DSC']:.6f} ± {official_sd['DSC']:.6f}")
print(f"- Case-level median DSC: {audit['DSC'].median():.6f}")
print(f"- DSC < 0.50: {(audit['DSC'] < 0.50).sum()} / {len(audit)} cases")
print(f"- HD95 > 20 mm: {(audit['HD95_mm'] > 20).sum()} / {len(audit)} cases")
print(f"- Smallest-burden quartile mean DSC: {burden_summary.iloc[0]['DSC_mean']:.6f}")
print(f"- Largest-burden quartile mean DSC: {burden_summary.iloc[-1]['DSC_mean']:.6f}")
print(f"- r(log lesion burden, DSC): {r_logburden_dsc:.4f}")


## Stage16 extended segmentation metrics — precision and lesion-level F1

The Stage16 evaluation additionally reports **precision** and **lesion-F1**. This section evaluates the already-frozen 93 Stage16 masks only; it performs **no training, neural inference, threshold tuning, or model selection**.

- **Precision** is computed per case as `TP / (TP + FP)` and averaged within each outer fold, matching the case-mean style of the primary Stage16 table.
- **Lesion F1 — any overlap** uses 26-connected lesion components and maximum bipartite one-to-one matching whenever predicted and expert lesions overlap.
- **Lesion F1 — IoU≥0.10** uses the same matching but requires component-pair IoU of at least 0.10.
- The headline extension is the equal-weight mean ± sample SD across the five outer-fold values.


In [ ]:
from scipy import ndimage
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import maximum_bipartite_matching

_STAGE16_STRUCT26 = np.ones((3,3,3), dtype=np.uint8)

def _stage16_label26(mask):
    return ndimage.label(mask, structure=_STAGE16_STRUCT26)[0]

def _stage16_lesion_counts(pred, gt, iou_threshold=0.0):
    pl = _stage16_label26(pred); gl = _stage16_label26(gt)
    npred, ngt = int(pl.max()), int(gl.max())
    if npred == 0 and ngt == 0: return 0, 0, 0
    if npred == 0: return 0, 0, ngt
    if ngt == 0: return 0, npred, 0
    both = (pl > 0) & (gl > 0)
    if not both.any(): return 0, npred, ngt
    pair = pl[both].astype(np.int64) * (ngt + 1) + gl[both].astype(np.int64)
    uniq, cnt = np.unique(pair, return_counts=True)
    pi, gi = uniq // (ngt + 1), uniq % (ngt + 1)
    ps = np.bincount(pl.ravel(), minlength=npred + 1)
    gs = np.bincount(gl.ravel(), minlength=ngt + 1)
    keep = []
    for pidx, gidx, inter in zip(pi, gi, cnt):
        ok = inter > 0 if iou_threshold <= 0 else inter / (ps[pidx] + gs[gidx] - inter) >= iou_threshold
        if ok: keep.append((int(pidx - 1), int(gidx - 1)))
    if not keep: return 0, npred, ngt
    rows = [x[0] for x in keep]; cols = [x[1] for x in keep]
    graph = csr_matrix((np.ones(len(rows), dtype=np.int8), (rows, cols)), shape=(npred, ngt))
    match = maximum_bipartite_matching(graph, perm_type='column')
    tp = int((match >= 0).sum())
    return tp, npred - tp, ngt - tp

def _stage16_lesion_f1(tp, fp, fn):
    den = 2 * tp + fp + fn
    return 1.0 if den == 0 else float(2 * tp / den)

extended_case_rows = []
for row in audit.sort_values(['fold','case']).itertuples(index=False):
    gt = load_bool(Path(row.gt_path))
    pred = load_bool(Path(row.mask_path))
    assert gt.shape == pred.shape, f'{row.case}: GT/pred shape mismatch'
    precision = 1.0 if (int(row.TP) + int(row.FP)) == 0 else float(int(row.TP)/(int(row.TP)+int(row.FP)))
    a = _stage16_lesion_counts(pred, gt, 0.0)
    s = _stage16_lesion_counts(pred, gt, 0.10)
    extended_case_rows.append({
        'case': str(row.case), 'fold': int(row.fold), 'Precision': precision,
        'LesionTP_any': a[0], 'LesionFP_any': a[1], 'LesionFN_any': a[2],
        'LesionTP_IoU010': s[0], 'LesionFP_IoU010': s[1], 'LesionFN_IoU010': s[2],
    })

extended_case = pd.DataFrame(extended_case_rows)
assert len(extended_case) == 93 and extended_case['case'].nunique() == 93

fold_extended_rows = []
for fold, g in extended_case.groupby('fold', sort=True):
    fold_extended_rows.append({
        'fold': int(fold), 'n_cases': int(len(g)),
        'Precision_case_mean': float(g['Precision'].mean()),
        'LesionF1_any_overlap': _stage16_lesion_f1(int(g.LesionTP_any.sum()), int(g.LesionFP_any.sum()), int(g.LesionFN_any.sum())),
        'LesionF1_IoU_ge_0_10': _stage16_lesion_f1(int(g.LesionTP_IoU010.sum()), int(g.LesionFP_IoU010.sum()), int(g.LesionFN_IoU010.sum())),
        'LesionTP_any': int(g.LesionTP_any.sum()), 'LesionFP_any': int(g.LesionFP_any.sum()), 'LesionFN_any': int(g.LesionFN_any.sum()),
        'LesionTP_IoU010': int(g.LesionTP_IoU010.sum()), 'LesionFP_IoU010': int(g.LesionFP_IoU010.sum()), 'LesionFN_IoU010': int(g.LesionFN_IoU010.sum()),
    })
stage16_extended_by_fold = pd.DataFrame(fold_extended_rows)
stage16_extended_summary = pd.DataFrame([
    {'Metric':'Precision_case_macro','Mean':float(stage16_extended_by_fold.Precision_case_mean.mean()),'Fold_SD':float(stage16_extended_by_fold.Precision_case_mean.std(ddof=1))},
    {'Metric':'LesionF1_any_overlap','Mean':float(stage16_extended_by_fold.LesionF1_any_overlap.mean()),'Fold_SD':float(stage16_extended_by_fold.LesionF1_any_overlap.std(ddof=1))},
    {'Metric':'LesionF1_IoU_ge_0_10','Mean':float(stage16_extended_by_fold.LesionF1_IoU_ge_0_10.mean()),'Fold_SD':float(stage16_extended_by_fold.LesionF1_IoU_ge_0_10.std(ddof=1))},
])
_expected={'Precision_case_macro':(0.7777367871281908,0.03918803792976655),'LesionF1_any_overlap':(0.7443683925600513,0.029949631773044393),'LesionF1_IoU_ge_0_10':(0.7302964674334995,0.03321153951229785)}
for r in stage16_extended_summary.itertuples(index=False):
    em,es=_expected[r.Metric]
    assert np.isclose(r.Mean,em,atol=1e-12) and np.isclose(r.Fold_SD,es,atol=1e-12), (r.Metric,r.Mean,r.Fold_SD)
display(stage16_extended_by_fold.style.format({'Precision_case_mean':'{:.6f}','LesionF1_any_overlap':'{:.6f}','LesionF1_IoU_ge_0_10':'{:.6f}'}))
display(stage16_extended_summary.style.format({'Mean':'{:.6f}','Fold_SD':'{:.6f}'}))
stage16_extended_by_fold.to_csv('/content/STAGE16_EXTENDED_SEGMENTATION_BY_FOLD.csv',index=False)
stage16_extended_summary.to_csv('/content/STAGE16_EXTENDED_SEGMENTATION_SUMMARY.csv',index=False)
print('Extended Stage16 closure:')
for r in stage16_extended_summary.itertuples(index=False):
    print(f'- {r.Metric}: {r.Mean:.6f} ± {r.Fold_SD:.6f}')
print('No training / new inference / retuning performed.')


## Multi-case and boundary-focused qualitative evidence

The analysis includes **diverse cases** and **boundary-focused cases**, selected reproducibly rather than by subjective visual preference.

**Different-case panel.** Six cases are selected from the frozen 93-case Stage16 table to cover:
- best, median and worst DSC,
- a successful case from the lowest 10% of expert lesion burden,
- the single lowest-burden case,
- and a strong high-burden case not already selected.

**Boundary-focused panel.** "Boundary case" is operationalized using **HD95**, the project's frozen boundary-distance metric:
- clean boundary: best DSC among cases at the minimum HD95,
- moderate boundary mismatch: case closest to the 75th percentile HD95,
- high-overlap / poor-boundary case: highest DSC among cases with HD95 > 20 mm,
- extreme boundary failure: maximum HD95.

Ground truth is used only for retrospective visualization and case selection. No training, inference, threshold tuning or model selection is performed.


In [ ]:
from scipy import ndimage
from matplotlib.patches import Patch

# ---------- deterministic case selection ----------
vis = audit.copy()
vis['GT_lesion_voxels'] = vis['TP'] + vis['FN']

used = set()

best_row = vis.loc[vis['DSC'].idxmax()]
median_row = vis.loc[(vis['DSC'] - vis['DSC'].median()).abs().idxmin()]
worst_row = vis.loc[vis['DSC'].idxmin()]
used.update([best_row['case'], median_row['case'], worst_row['case']])

low10_cut = vis['GT_lesion_voxels'].quantile(0.10, interpolation='higher')
low10 = vis[vis['GT_lesion_voxels'] <= low10_cut].copy()
low_success = low10.loc[low10['DSC'].idxmax()]

lowest_burden = vis.loc[vis['GT_lesion_voxels'].idxmin()]

high10_cut = vis['GT_lesion_voxels'].quantile(0.90, interpolation='lower')
high10 = vis[(vis['GT_lesion_voxels'] >= high10_cut) & (~vis['case'].isin(used))].copy()
high_strong = high10.loc[high10['DSC'].idxmax()]

different_rows = pd.DataFrame(
    [best_row, median_row, worst_row, low_success, lowest_burden, high_strong],
    index=['Best overall','Median case','Worst overall','Low-burden success','Lowest-burden case','High-burden strong']
)

min_hd = float(vis['HD95_mm'].min())
clean_pool = vis[np.isclose(vis['HD95_mm'], min_hd)]
clean_boundary = clean_pool.loc[clean_pool['DSC'].idxmax()]

q75_hd = float(vis['HD95_mm'].quantile(0.75))
moderate_boundary = vis.loc[(vis['HD95_mm'] - q75_hd).abs().idxmin()]

poor_boundary_pool = vis[vis['HD95_mm'] > 20].copy()
high_dsc_poor_boundary = poor_boundary_pool.loc[poor_boundary_pool['DSC'].idxmax()]

extreme_boundary = vis.loc[vis['HD95_mm'].idxmax()]

boundary_rows = pd.DataFrame(
    [clean_boundary, moderate_boundary, high_dsc_poor_boundary, extreme_boundary],
    index=['Clean boundary','Moderate boundary mismatch','High DSC / poor boundary','Extreme boundary failure']
)

print('Different-case selection')
display(different_rows[['case','fold','DSC','HD95_mm','Sensitivity','GT_lesion_voxels']])
print('Boundary-focused selection')
display(boundary_rows[['case','fold','DSC','HD95_mm','Sensitivity','GT_lesion_voxels']])

# Hard-lock the actual outputs of the deterministic selectors for this frozen table.
expected_different = [
    'MSLesSeg_P4_T3','MSLesSeg_P6_T1','MSLesSeg_P18_T1',
    'MSLesSeg_P16_T1','MSLesSeg_P20_T1','MSLesSeg_P7_T2'
]
expected_boundary = [
    'MSLesSeg_P4_T3','MSLesSeg_P11_T1','MSLesSeg_P21_T1','MSLesSeg_P3_T2'
]
assert different_rows['case'].tolist() == expected_different, different_rows['case'].tolist()
assert boundary_rows['case'].tolist() == expected_boundary, boundary_rows['case'].tolist()

# Validate the selection rules themselves as an additional integrity guard.
assert low_success['case'] in set(low10['case'])
assert np.isclose(float(low_success['DSC']), float(low10['DSC'].max()))
assert int(lowest_burden['GT_lesion_voxels']) == int(vis['GT_lesion_voxels'].min())
assert high_strong['case'] in set(high10['case'])
assert np.isclose(float(high_strong['DSC']), float(high10['DSC'].max()))
assert np.isclose(float(clean_boundary['HD95_mm']), min_hd)
assert np.isclose(
    abs(float(moderate_boundary['HD95_mm']) - q75_hd),
    float((vis['HD95_mm'] - q75_hd).abs().min())
)
assert float(high_dsc_poor_boundary['HD95_mm']) > 20
assert np.isclose(float(high_dsc_poor_boundary['DSC']), float(poor_boundary_pool['DSC'].max()))
assert np.isclose(float(extreme_boundary['HD95_mm']), float(vis['HD95_mm'].max()))

# ---------- visualization helpers ----------
def _boundary2d(mask):
    if not mask.any():
        return np.zeros_like(mask, dtype=bool)
    er = ndimage.binary_erosion(mask, structure=np.ones((3,3), dtype=bool), border_value=0)
    return mask & (~er)

def _best_gt_slice(gt):
    return int(np.argmax(gt.sum(axis=(0,1))))

def _boundary_slice(gt, pred):
    scores = []
    for z in range(gt.shape[2]):
        g = gt[:,:,z]
        p = pred[:,:,z]
        if not (g.any() or p.any()):
            scores.append(-1.0)
            continue
        disagreement = np.logical_xor(_boundary2d(g), _boundary2d(p)).sum()
        support = min(g.sum(), 500) / 500.0
        scores.append(float(disagreement) + 15.0*float(support))
    return int(np.argmax(scores))

def _error2d(gt2, pred2):
    e = np.zeros(gt2.shape, dtype=np.uint8)
    e[gt2 & pred2] = 1
    e[(~gt2) & pred2] = 2
    e[gt2 & (~pred2)] = 3
    return e

def _case_arrays(row):
    case = row['case']
    gt_path = Path(row['gt_path'])
    pred_path = Path(row['mask_path'])
    flair_path = gt_path.parent.parent / 'imagesTr' / f'{case}_0000.nii.gz'
    for p in [gt_path, pred_path, flair_path]:
        assert p.exists(), f'Missing required file: {p}'
    gt = load_bool(gt_path)
    pred = load_bool(pred_path)
    flair = robust_flair(flair_path)
    assert gt.shape == pred.shape == flair.shape
    return flair, gt, pred

def _display_xy(a):
    return a.T

def _boundary_overlay(flair2, gt2, pred2):
    rgb = np.dstack([flair2, flair2, flair2])
    bg = _boundary2d(gt2)
    bp = _boundary2d(pred2)
    rgb[bg] = [1.0, 0.84, 0.0]     # expert boundary
    rgb[bp] = [0.0, 0.85, 1.0]     # predicted boundary
    rgb[bg & bp] = [1.0, 1.0, 1.0]
    return rgb

def _crop_box(mask, pad=18):
    yy, xx = np.where(mask)
    if len(xx) == 0:
        return (slice(None), slice(None))
    y0, y1 = max(0, int(yy.min())-pad), min(mask.shape[0], int(yy.max())+pad+1)
    x0, x1 = max(0, int(xx.min())-pad), min(mask.shape[1], int(xx.max())+pad+1)
    return (slice(y0,y1), slice(x0,x1))

error_cmap = ListedColormap(['black','#2ca02c','#d62728','#1f77b4'])

# ---------- panel A: six different cases ----------
fig, axs = plt.subplots(6, 4, figsize=(13.5,17))
for j, title in enumerate(['FLAIR','Expert GT','Frozen prediction','TP / FP / FN']):
    axs[0,j].set_title(title, fontsize=12, fontweight='bold')

for r, (label, row) in enumerate(different_rows.iterrows()):
    flair, gt, pred = _case_arrays(row)
    z = _best_gt_slice(gt)
    f2 = _display_xy(flair[:,:,z])
    g2 = _display_xy(gt[:,:,z])
    p2 = _display_xy(pred[:,:,z])
    e2 = _error2d(g2,p2)

    axs[r,0].imshow(f2, cmap='gray', origin='lower')
    axs[r,1].imshow(g2, cmap='gray', vmin=0, vmax=1, origin='lower')
    axs[r,2].imshow(p2, cmap='gray', vmin=0, vmax=1, origin='lower')
    axs[r,3].imshow(e2, cmap=error_cmap, vmin=0, vmax=3, interpolation='nearest', origin='lower')

    axs[r,0].text(
        -0.08, 0.5,
        f"{label}\n{row['case']}\nDSC {row['DSC']:.3f}\n"
        f"HD95 {row['HD95_mm']:.2f} mm\nGT {int(row['GT_lesion_voxels']):,} vox",
        transform=axs[r,0].transAxes, ha='right', va='center',
        fontsize=8.5, fontweight='bold'
    )
    for a in axs[r]:
        a.axis('off')

fig.legend(
    handles=[
        Patch(facecolor='#2ca02c', label='TP'),
        Patch(facecolor='#d62728', label='FP'),
        Patch(facecolor='#1f77b4', label='FN'),
    ],
    loc='lower center', ncol=3, frameon=False
)
fig.suptitle(
    'GraphMS v3.5.1 — Multi-case qualitative segmentation evidence\n'
    'Representative performance and lesion-burden diversity; frozen development OOF masks',
    fontsize=14, fontweight='bold', y=.995
)
fig.tight_layout(rect=[.12,.03,1,.98])
multi_path = Path('/content/SEGMENTATION_MULTI_CASE_QUALITATIVE.png')
fig.savefig(multi_path, dpi=240, bbox_inches='tight')
plt.show()

# ---------- panel B: HD95-driven boundary cases ----------
fig, axs = plt.subplots(4, 3, figsize=(13.5,14))
for j, title in enumerate([
    'Full FLAIR + GT / prediction boundaries',
    'Boundary-region zoom',
    'TP / FP / FN zoom'
]):
    axs[0,j].set_title(title, fontsize=11, fontweight='bold')

for r, (label, row) in enumerate(boundary_rows.iterrows()):
    flair, gt, pred = _case_arrays(row)
    z = _boundary_slice(gt, pred)

    f2 = _display_xy(flair[:,:,z])
    g2 = _display_xy(gt[:,:,z])
    p2 = _display_xy(pred[:,:,z])
    ov = _boundary_overlay(f2, g2, p2)
    e2 = _error2d(g2,p2)
    crop = _crop_box(g2 | p2, pad=18)

    axs[r,0].imshow(ov, origin='lower')
    axs[r,1].imshow(ov[crop], origin='lower')
    axs[r,2].imshow(e2[crop], cmap=error_cmap, vmin=0, vmax=3, interpolation='nearest', origin='lower')

    axs[r,0].text(
        -0.08, 0.5,
        f"{label}\n{row['case']}\nDSC {row['DSC']:.3f}\n"
        f"HD95 {row['HD95_mm']:.2f} mm\nslice z={z}",
        transform=axs[r,0].transAxes, ha='right', va='center',
        fontsize=8.5, fontweight='bold'
    )
    for a in axs[r]:
        a.axis('off')

fig.legend(
    handles=[
        Patch(facecolor='#ffd700', label='Expert boundary'),
        Patch(facecolor='#00d9ff', label='Predicted boundary'),
        Patch(facecolor='white', edgecolor='black', label='Boundary overlap'),
        Patch(facecolor='#2ca02c', label='TP'),
        Patch(facecolor='#d62728', label='FP'),
        Patch(facecolor='#1f77b4', label='FN'),
    ],
    loc='lower center', ncol=3, frameon=False
)
fig.suptitle(
    'GraphMS v3.5.1 — Boundary-focused segmentation cases\n'
    'HD95-driven examples showing agreement, moderate mismatch, and boundary failure',
    fontsize=14, fontweight='bold', y=.995
)
fig.tight_layout(rect=[.15,.06,1,.97])
boundary_path = Path('/content/SEGMENTATION_BOUNDARY_FOCUSED.png')
fig.savefig(boundary_path, dpi=240, bbox_inches='tight')
plt.show()

selection_rows = []
for label, row in different_rows.iterrows():
    selection_rows.append({
        'panel':'multi_case','selection':label,'case':row['case'],'fold':int(row['fold']),
        'DSC':float(row['DSC']),'HD95_mm':float(row['HD95_mm']),
        'Sensitivity':float(row['Sensitivity']),'GT_lesion_voxels':int(row['GT_lesion_voxels'])
    })
for label, row in boundary_rows.iterrows():
    selection_rows.append({
        'panel':'boundary','selection':label,'case':row['case'],'fold':int(row['fold']),
        'DSC':float(row['DSC']),'HD95_mm':float(row['HD95_mm']),
        'Sensitivity':float(row['Sensitivity']),'GT_lesion_voxels':int(row['GT_lesion_voxels'])
    })
selection_path = Path('/content/SEGMENTATION_QUALITATIVE_CASE_SELECTION.csv')
pd.DataFrame(selection_rows).to_csv(selection_path, index=False)

print('Saved:', multi_path)
print('Saved:', boundary_path)
print('Saved:', selection_path)



## What the diagnostic can establish

This analysis characterizes **where** the finalized development system fails; it does not retroactively change the model. In particular, if the lowest-DSC cases cluster at low expert lesion burden and are FN-dominant, the defensible interpretation is that **low-lesion-burden / missed-lesion sensitivity is a principal residual weakness of the frozen segmentation system**. That is a development-set failure-mode observation, not a clinical-generalization claim.

The appropriate interpretation is therefore to report this limitation transparently alongside the strong median/high-performing cases and the leakage-controlled five-fold aggregate—not to modify the frozen preprocessing or tune Stage11 again on the same outer-fold evidence.



## Same-cohort baseline context

This comparison summarizes the available same-cohort development evidence **without retraining or retuning**.

The historical project branch scoreboard reports **series/case-weighted mean DSC**, so the final Hybrid is shown here using its corresponding 93-case arithmetic mean rather than substituting the official equal-fold headline. The official Stage16 result remains the primary result elsewhere in this notebook.

These values are internal development comparison evidence, not external validation and not a claim that the final graph-aware Hybrid numerically dominates the CNN reference.


In [ ]:
baseline_compare = pd.DataFrame([
    ["ResEncM-250 raw", "93-series five-fold development OOF", 0.749353, "Completed CNN reference"],
    ["ResEncM-250 + component filter", "93-series five-fold development OOF", 0.749416, "Completed CNN reference"],
    ["ResEncM-250 + filter + opening", "93-series five-fold development OOF", 0.725885, "Morphology evaluated; worse"],
    ["CATMIL M150 raw", "93-series five-fold development OOF", 0.744290, "Completed auxiliary"],
    ["Cross-fit calibrated CNN", "93-series five-fold development OOF", 0.745780, "Completed auxiliary"],
    ["Stage7 GAT final", "93-series nested development OOF", 0.744531, "Evaluated; not promoted"],
    ["GraphMS v3.5.1 Hybrid", "93-case frozen Stage16 OOF; case-weighted descriptive mean", float(audit["DSC"].mean()), "Final selected graph-aware system"],
], columns=["Branch", "Evaluation scope", "DSC", "Status"])
display(baseline_compare.style.format({"DSC": "{:.6f}"}))
baseline_compare.to_csv("/content/STAGE16_SAME_COHORT_BASELINE_COMPARISON.csv", index=False)
print("Primary official GraphMS Stage16 result remains:")
print(f"DSC = {official_mean['DSC']:.12f} ± {official_sd['DSC']:.12f} (equal-weight mean ± sample SD across fold means)")
print("No numerical-superiority claim over the CNN reference is made.")


## 16-stage implementation matrix

Every pipeline stage is assigned a final implementation status. Optional, historical, unavailable, or rejected alternatives are **not relabeled as implemented components**.


In [ ]:
stage_matrix = pd.DataFrame([
    [1, "Data acquisition", "MSLesSeg development cohort; FLAIR/T1/T2 and expert lesion masks", "IMPLEMENTED / FINALIZED"],
    [2, "Preprocessing", "Co-registered multimodal inputs with finalized nnU-Net plan-based runtime preprocessing and geometry validation", "IMPLEMENTED / FINALIZED"],
    [3, "Data augmentation", "Rotation / flip / scale / elastic / gamma-noise / crop in training lineage", "IMPLEMENTED — TRAINING ONLY"],
    [4, "Multimodal fusion", "FLAIR + T1 + T2 channel-wise input; PD unavailable", "IMPLEMENTED / DATASET-CONSTRAINED"],
    [5, "CNN feature extraction", "ResEncM-250 residual nnU-Net backbone + multiscale features", "IMPLEMENTED"],
    [6, "Graph construction", "4³ patch/grid nodes; 6-neighbour + spatial kNN8 + feature kNN4 + self-loops", "IMPLEMENTED — PATCH/kNN"],
    [7, "GNN learning", "Two-layer edge-aware TrueGAT, hidden 128, four heads", "IMPLEMENTED"],
    [8, "Hybrid feature fusion", "CNN/GNN concat + SE + self-attention + multi-scale fusion", "IMPLEMENTED"],
    [9, "Lesion segmentation head", "Transposed-convolution decoder + skips + 1×1×1 lesion output", "IMPLEMENTED"],
    [10, "Segmentation loss", "Dice + BCE", "IMPLEMENTED; FOCAL NOT PROMOTED"],
    [11, "Post-processing", "Cross-fitted threshold + 26-CCA + remove components <10 voxels", "IMPLEMENTED; MORPHOLOGY EVALUATED / NOT PROMOTED"],
    [12, "Lesion features", "Volume/count/shape + GLCM + multimodal/spatial context", "IMPLEMENTED — STAGE12 v2.1"],
    [13, "Risk prediction", "MRI_SPATIAL_SVM + MRI_SPATIAL_RIDGE from finalized Stage12 imaging features", "IMPLEMENTED RESEARCH OUTPUT"],
    [14, "Multi-task learning", "Historical joint Dice+BCE + CE branch", "IMPLEMENTED / EVALUATED / AUXILIARY / NOT PROMOTED"],
    [15, "Model training", "AdamW + CosineAnnealingLR + dropout 0.30 + weight decay 1e-4", "COMPLETED PROVENANCE"],
    [16, "Evaluation", "DSC/IoU/Sens/Spec/HD95/precision/lesion-F1 + downstream risk metrics", "COMPLETE / FINALIZED"],
], columns=["Stage", "Pipeline component", "Final project disposition", "Implementation status"])
assert stage_matrix['Stage'].tolist() == list(range(1,17))
assert stage_matrix['Stage'].is_unique
assert len(stage_matrix) == 16

display(stage_matrix)
stage_matrix.to_csv("/content/STAGE_IMPLEMENTATION_MATRIX.csv", index=False)
print("All 16 stages are traceable to implemented or evaluated evidence.")
print("New pretraining, hard-negative/small-lesion training and external validation remain future experiments, not backfilled claims.")


In [ ]:
# Export a compact segmentation-evaluation bundle.
from zipfile import ZipFile, ZIP_DEFLATED

artifacts = [
    '/content/SEGMENTATION_COHORT_SUMMARY.csv',
    '/content/SEGMENTATION_FOLD_CASE_MEANS.csv',
    '/content/SEGMENTATION_BURDEN_QUARTILES.csv',
    '/content/SEGMENTATION_WORST10.csv',
    '/content/SEGMENTATION_DSC_DISTRIBUTION.png',
    '/content/SEGMENTATION_HD95_DISTRIBUTION.png',
    '/content/SEGMENTATION_DSC_BY_FOLD.png',
    '/content/SEGMENTATION_DSC_VS_LESION_BURDEN.png',
    '/content/SEGMENTATION_DSC_BY_BURDEN_QUARTILE.png',
    '/content/STAGE16_SAME_COHORT_BASELINE_COMPARISON.csv',
    '/content/STAGE_IMPLEMENTATION_MATRIX.csv',
    '/content/SEGMENTATION_MULTI_CASE_QUALITATIVE.png',
    '/content/SEGMENTATION_BOUNDARY_FOCUSED.png',
    '/content/SEGMENTATION_QUALITATIVE_CASE_SELECTION.csv',
    '/content/STAGE16_EXTENDED_SEGMENTATION_BY_FOLD.csv',
    '/content/STAGE16_EXTENDED_SEGMENTATION_SUMMARY.csv',
]

for label, row in selected.iterrows():
    artifacts.append(f"/content/SEGMENTATION_{label}_{row['case']}.png")

artifacts = [Path(x) for x in artifacts]
missing = [str(x) for x in artifacts if not x.exists()]
empty = [str(x) for x in artifacts if x.exists() and x.stat().st_size == 0]
if missing or empty:
    raise RuntimeError(f'Diagnostics export incomplete. Missing={missing}; empty={empty}')
assert len(artifacts) == 19, f'Expected 19 diagnostic artifacts, found {len(artifacts)}'

zip_path = Path('/content/GraphMS_Segmentation_Diagnostics.zip')
with ZipFile(zip_path, 'w', compression=ZIP_DEFLATED) as zf:
    for a in artifacts:
        zf.write(a, arcname=a.name)

print('Diagnostics bundle:', zip_path)
print('Included files:', len(artifacts))




